## Basic Generate

In [1]:
import httpx

with httpx.Client(
    proxy=None,
    trust_env=False
) as client:
    response = client.post(
        "http://127.0.0.1:8000/basic_generate",
        json={
            "prompt": "What is the capital of the United States?"
        },
        timeout=60.0
    )

print(response.status_code)
print(response.text)

200
{"generated_text":"What is the capital of the United States? The capital of the United States is Washington, D.C. (also known as the District of Columbia). It serves as the seat of government for the 50 states and the federal government of the United States.\n\nWashington, D.C., located in"}


## Batch Generate

In [2]:
import httpx

payload = {
    "prompts": [
        "The capital of France is",
        "The capital of India is",
        "The capital of Japan is",
        "The largest planet is"
    ]
}

with httpx.Client(trust_env=False) as client:
    response = client.post(
        "http://127.0.0.1:8000/generate",
        json=payload,
        timeout=60.0
    )

print(response.status_code)
print(response.json())

200
{'generated_texts': ["The capital of France is ________.\nA. Paris\nB. London\nC. Berlin\nD. Moscow\n\nTo determine the capital of France, let's follow a step-by-step analysis:\n\n1. **Identify the Capital of France**: The capital of France is", 'The capital of India is located on the\nA. Arabian Sea coast\nB. Bay of Bengal coast\nC. Indus River delta\nD. Ganges Delta\n\nThe capital of India is located on the **Ganges Delta**.\n\nTo elaborate:\n\n- The Indian', "The capital of Japan is located in the ___\nA. East China Sea\nB. Pacific Ocean\nC. Sea of Japan\nD. Bashi Channel\n\nTo determine the correct answer, let's analyze each option step by step:\n\n1. **East China Sea**:", 'The largest planet isIf you were to rank the planets in order of size, from largest to smallest, which would be at the top?\nTo determine the ranking of the planets from largest to smallest based on their sizes, we need to consider the following:\n\n1. The']}


## Concurrent Request from multiple clients

In [3]:
import asyncio
import httpx
import time

prompts = [
    "What is KV cache?",
    "What is attention?",
    "What is speculative decoding?",
    "What is continuous batching?"
]

async def call_api(prompt):
    async with httpx.AsyncClient(trust_env=False) as client:
        start = time.perf_counter()

        r = await client.post(
            "http://127.0.0.1:8000/basic_generate",
            json={"prompt": prompt},
            timeout=60.0
        )

        elapsed = time.perf_counter() - start

        return {
            "prompt": prompt,
            "time": round(elapsed, 2),
            "response": r.json()
        }

results = await asyncio.gather(
    *[call_api(p) for p in prompts]
)

for r in results:
    print(r)

{'prompt': 'What is KV cache?', 'time': 1.53, 'response': {'generated_text': 'What is KV cache? What are its benefits and drawbacks?\nKV (Key-Value) Cache is a data structure used to store key-value pairs, where the keys are unique identifiers or labels that can be used to quickly locate their corresponding values. It is often used in applications'}}
{'prompt': 'What is attention?', 'time': 2.93, 'response': {'generated_text': 'What is attention? How does it affect the brain?\n\nAttention is a mental process that involves focusing on one thing at a time. It helps us to stay focused, pay attention to what we are doing and make decisions based on our priorities. Attention can be affected by various'}}
{'prompt': 'What is speculative decoding?', 'time': 4.3, 'response': {'generated_text': 'What is speculative decoding? What are the key features of a good speculative decoder?\n\nHow can I make a good speculative decoder?\n\nChoose your answer. Are these two questions paraphrases of each ot

## Latency - KV Cache (same question - multiple iteration)
1. Post first time, the response is faster

In [ ]:
# import time
# import httpx

# with httpx.Client(trust_env=False) as client:
#     start = time.perf_counter()

#     response = client.post(
#         "http://127.0.0.1:8000/basic_generate",
#         json={
#             "prompt": "Explain KV cache in 3 sentences."
#         },
#         timeout=60.0
#     )

#     end = time.perf_counter()

# print("Latency:", round(end - start, 3), "seconds")
# print(response.json())

In [5]:
import json
import httpx

prompt = "What is the capital of the United States?"
tokens = []
sequence_ids = set()

with httpx.Client(trust_env=False) as client:
    with client.stream(
        "POST",
        "http://127.0.0.1:8000/generate_stream",
        json={"prompt": prompt},
        timeout=120.0,
    ) as response:
        print("status:", response.status_code)
        for line in response.iter_lines():
            if not line or not line.startswith("data: "):
                continue
            data = json.loads(line[6:])
            token = data["token"]
            seq_id = data["sequence_id"]
            tokens.append(token)
            sequence_ids.add(seq_id)
            print(token, end="", flush=True)

print("\n")
print("sequence_id:", sequence_ids)
print("num tokens:", len(tokens))
print("full text:", prompt + "".join(tokens))

status: 200
 Washington, D.C. is the capital of the United States. It is located on the National Mall in

sequence_id: {'bf1b87d8-a261-481e-9e2a-08831607bb11'}
num tokens: 21
full text: What is the capital of the United States? Washington, D.C. is the capital of the United States. It is located on the National Mall in
